# TELUGU DEVOTEE — Wan2.1 scene-motion generator

This notebook reads the existing Episode 2 Markdown directly from the TELUGU DEVOTEE GitHub repository, extracts each scene's `IMAGE PROMPT`, and generates silent AI motion clips with the open Wan2.1 T2V 1.3B model. It saves each finished clip to Google Drive so a disconnected free session can resume later.

**Safety:** this notebook has no YouTube credentials or upload code. It creates review files only. Free GPU access is not guaranteed. Start with one scene before attempting batches.

In [ ]:
%pip -q install -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg requests

In [ ]:
import os, re, gc, requests, torch
from pathlib import Path
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab, choose Runtime > Change runtime type > T4 GPU if offered. Do not switch to paid services.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/TELUGU_DEVOTEE/episode2')
VIDEO_DIR = BASE / 'scene_videos'
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_URL = 'https://raw.githubusercontent.com/Gopi-chand1/telugu-devotee-automation/main/output/episode_package.md'
response = requests.get(MARKDOWN_URL, timeout=60)
response.raise_for_status()
markdown = response.text
print('Downloaded episode Markdown:', len(markdown), 'characters')

scene_parts = re.split(r'(?m)^###\\s*SCENE\\s+(\\d+)\\s*\\|[^\\n]*\\n', markdown)
scenes = {}
for i in range(1, len(scene_parts), 2):
    number = int(scene_parts[i])
    block = scene_parts[i + 1]
    match = re.search(r'(?ms)^IMAGE PROMPT:\\s*(.*?)(?=^MOTION PROMPT:|\\Z)', block)
    if match and match.group(1).strip():
        scenes[number] = match.group(1).strip()
print('Scene image prompts found:', len(scenes))
if len(scenes) < 1:
    raise RuntimeError('No IMAGE PROMPT blocks found. Check the episode Markdown format.')

# Safe first test: scene 1 only. After reviewing it, change the range for small batches.
START_SCENE = 1
END_SCENE = 1
# Wan frame counts should follow 4*k+1. 41 frames ≈ 2.56 seconds at 16 fps.
# Increase to 81 (~5.06s) only after the first test succeeds and memory allows it.
NUM_FRAMES = 41
STEPS = 8
FPS = 16
print(f'Planned scenes {START_SCENE}-{END_SCENE}; frames={NUM_FRAMES}; output={VIDEO_DIR}')

In [ ]:
from diffusers import WanPipeline
from diffusers.utils import export_to_video

model_id = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
pipe = WanPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
try:
    pipe.vae.enable_tiling()
except Exception:
    pass
print('Wan2.1 loaded. First model download can take several minutes.')

In [ ]:
from IPython.display import Video, display

completed, failed = [], []
for number in range(START_SCENE, END_SCENE + 1):
    if number not in scenes:
        print(f'Scene {number}: no image prompt; skipped')
        failed.append(number)
        continue
    output_path = VIDEO_DIR / f'scene_{number:03d}.mp4'
    if output_path.exists() and output_path.stat().st_size > 100_000:
        print(f'Scene {number}: already exists; resume-safe skip')
        completed.append(str(output_path))
        continue
    print(f'Generating scene {number}...')
    try:
        with torch.inference_mode():
            result = pipe(
                prompt=scenes[number],
                negative_prompt='blurry, low quality, distorted face, extra fingers, extra limbs, deformed hands, duplicated people, flicker, jitter, warped body, plastic skin, modern objects, text, watermark, subtitles, logo',
                height=480, width=832, num_frames=NUM_FRAMES,
                num_inference_steps=STEPS, guidance_scale=5.0,
            )
        export_to_video(result.frames[0], str(output_path), fps=FPS)
        if not output_path.exists() or output_path.stat().st_size < 100_000:
            raise RuntimeError('Output MP4 missing or unexpectedly small')
        completed.append(str(output_path))
        print(f'Scene {number}: PASS — {output_path} ({output_path.stat().st_size:,} bytes)')
        display(Video(str(output_path), embed=True, width=480))
        del result
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as exc:
        print(f'Scene {number}: FAILED — {type(exc).__name__}: {exc}')
        failed.append(number)
        gc.collect()
        torch.cuda.empty_cache()

print('\nBatch finished.')
print('Completed:', len(completed), 'Failed/skipped:', failed)
print('Saved in Google Drive:', VIDEO_DIR)
print('No audio muxing and no YouTube upload were performed.')

## After the first test

1. Review `scene_001.mp4` for motion quality, faces, hands, flicker, and character continuity.
2. If it succeeds, change `START_SCENE` and `END_SCENE` to a small range (for example, 2–3), then rerun the setup/configuration cell and generation cell as needed. Existing files are skipped automatically.
3. Keep batches small. A free Colab GPU can disconnect, and 60 scenes may take many sessions. The default test is only about 2.6 seconds per scene; it is not yet the required 10-second finished scene.
4. The next stage is to combine reviewed scene clips with Telugu WAV files and create a final MP4 review artifact. Public publishing remains disabled.